In [18]:
import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [19]:
# File paths
comparison_path = f"../output/MERSCOPE_WT_AD_comparison/"

In [20]:
# Enhance spatial resolution
def subdivide_spots_50_to_25_um(spots_raw: "sc.AnnData", sub_um: float = 25.0) -> "sc.AnnData":
    """Each 50×50 μm spot becomes four 25×25 μm sub-spots; counts are split evenly."""
    half_sub = sub_um / 2
    # Centroid offsets (μm) from parent center: SW, SE, NW, NE in (x, y)
    offsets = np.array(
        [
            [-half_sub, -half_sub],
            [half_sub, -half_sub],
            [-half_sub, half_sub],
            [half_sub, half_sub],
        ]
    )
    n = spots_raw.n_obs
    gx = spots_raw.obs["global_x"].to_numpy(dtype=float)
    gy = spots_raw.obs["global_y"].to_numpy(dtype=float)
    new_gx = np.column_stack([gx + offsets[i, 0] for i in range(4)]).ravel()
    new_gy = np.column_stack([gy + offsets[i, 1] for i in range(4)]).ravel()
    ix = np.arange(n).repeat(4)
    X = spots_raw.X[ix].copy()
    if hasattr(X, "multiply"):
        X = X.multiply(1.0 / 4.0)
    else:
        X = np.asarray(X, dtype=np.float64) / 4.0
    inherited = ["region_labels", "brain_area", "batch"]
    obs_new = spots_raw.obs.iloc[ix][inherited].reset_index(drop=True)
    obs_new["global_x"] = new_gx
    obs_new["global_y"] = new_gy
    parent_ids = spots_raw.obs["spot_id"].astype(str).to_numpy()[ix]
    quad = np.tile(np.arange(4), n)
    obs_new["spot_id"] = parent_ids + "_25um_q" + quad.astype(str)
    out = anndata.AnnData(X=X, obs=obs_new, var=spots_raw.var.copy())
    if spots_raw.uns:
        out.uns = dict(spots_raw.uns)
    return out

spots = subdivide_spots_50_to_25_um(spots_raw)
spots

AnnData object with n_obs × n_vars = 121084 × 290
    obs: 'region_labels', 'brain_area', 'batch', 'global_x', 'global_y', 'spot_id'
    var: 'genes'
    uns: 'brain_area_colors'

In [ ]:
# ==================== Read data ==================== #

# -------------------- Spots -------------------- #
spots_WT = sc.read_h5ad(f"../data/MERSCOPE_WT_1/processed_data/spots.h5ad")
spots_AD = sc.read_h5ad(f"../data/MERSCOPE_AD_1/processed_data/spots.h5ad")

# Adjust coordinates
spots_WT.obs["global_x_adjusted"] = spots_WT.obs["global_y_new"].copy()
spots_WT.obs["global_y_adjusted"] = spots_WT.obs["global_x_new"].copy()
spots_WT.obs["global_x"] = spots_WT.obs["global_x_adjusted"].copy()
spots_WT.obs["global_y"] = spots_WT.obs["global_y_adjusted"].copy()

spots_AD.obs["global_x_adjusted"] = spots_AD.obs["global_x_new"].copy()
spots_AD.obs["global_y_adjusted"] = spots_AD.obs["global_y_new"].copy()
spots_AD.obs["global_x_adjusted"] += 12000
spots_AD.obs["global_y_adjusted"] += 7200
spots_AD.obs["global_x"] = spots_AD.obs["global_x_adjusted"].copy()
spots_AD.obs["global_y"] = spots_AD.obs["global_y_adjusted"].copy()

# Merge spots
spots_dict = {"MERSCOPE_WT_1": spots_WT, "MERSCOPE_AD_1": spots_AD}
spots_raw = anndata.concat(spots_dict, axis = 0, merge = "same", label = "batch")
spots_raw

# -------------------- Cells -------------------- #
adata_1 = sc.read_h5ad(f"../data/MERSCOPE_WT_1/processed_data/adata.h5ad")
adata_2 = sc.read_h5ad(f"../data/MERSCOPE_AD_1/processed_data/adata.h5ad")

# Adjust coordinates
adata_1.obs["global_x_adjusted"] = adata_1.obs["global_y_new"].copy()
adata_1.obs["global_y_adjusted"] = adata_1.obs["global_x_new"].copy()
adata_1.obs["global_x"] = adata_1.obs["global_x_adjusted"].copy()
adata_1.obs["global_y"] = adata_1.obs["global_y_adjusted"].copy()

adata_2.obs["global_x_adjusted"] = adata_2.obs["global_x_new"].copy()
adata_2.obs["global_y_adjusted"] = adata_2.obs["global_y_new"].copy()
adata_2.obs["global_x_adjusted"] += 12000
adata_2.obs["global_y_adjusted"] += 7200
adata_2.obs["global_x"] = adata_2.obs["global_x_adjusted"].copy()
adata_2.obs["global_y"] = adata_2.obs["global_y_adjusted"].copy()

# Merge cells
adata_dict = {"MERSCOPE_WT_1": adata_1, "MERSCOPE_AD_1": adata_2}
adata = anndata.concat(adata_dict, axis = 0, merge = "same", label = "batch")
adata

# Neurons only
adata_neuron = adata[adata.obs["cell_type"].isin(["Glutamatergic", "GABAergic"])].copy()

# -------------------- Granules -------------------- #
granule_adata = sc.read_h5ad(comparison_path + "granule_adata_tsne.h5ad")
granule_subtype_df = pd.read_parquet(comparison_path + "granule_subtype_labels_granule_adata_tsne.parquet")
granule_adata.obs["granule_subtype_kmeans"] = granule_subtype_df["granule_subtype_kmeans"].astype("category")
granule_adata.obs["granule_subtype_manual"] = granule_adata.obs["granule_subtype_manual"].astype("category")
granule_adata.obs["granule_subtype"] = granule_adata.obs["granule_subtype_manual_simple"].astype("category")